# Fine-tuning GPT-2 with LoRA for text rewriting

I instruction-tuned GPT-2 with LoRA on the smol-rewrite part of the SmolTalk dataset. I added my own chat tokens and only computed the loss on the assistant part of each conversation. There is no real evaluation yet, just the training loss and one generated rewrite compared with the original.

In [1]:
!pip install --upgrade torchao

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, TaskType, get_peft_model, LoraConfig

# Load the GPT-2 tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
tokenizer.add_special_tokens({"pad_token" : "<pad>",
                              "bos_token" : "<start>",
                              "eos_token" : "<end>",
                              "additional_special_tokens" : ["|system|", "|user|", "|assistant|"]})
tokenizer.padding_side = "right"
model.resize_token_embeddings(len(tokenizer))

Embedding(50263, 768)

In [5]:
from datasets import load_dataset

full_dataset = load_dataset(
    "HuggingFaceTB/smoltalk",
    name="smol-rewrite",
    split="train"
)

test_dataset = load_dataset(
    "HuggingFaceTB/smoltalk",
    name="smol-rewrite",
    split="test"
)


# 2. Slice off exactly 15,000 rows right away to keep it at your ~30 MB target
train_dataset = full_dataset.select(range(3000))
test_dataset = test_dataset.select(range(100))

README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

data/smol-rewrite/train-00000-of-00001.p(…):   0%|          | 0.00/38.9M [00:00<?, ?B/s]

data/smol-rewrite/test-00000-of-00001.pa(…):   0%|          | 0.00/2.06M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/53342 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2808 [00:00<?, ? examples/s]

In [6]:
from torch.utils.data import Dataset
import torch

class CustDataset(Dataset):
  def __init__(self, dataset):
    self.dataset = dataset

  def __len__(self):
    return len(self.dataset)

  def __getitem__(self, idx):
    msg = self.dataset[idx]['messages']

    full_text = (
        f"<start>|system| {msg[0]['content']} <end>"
        f"<start>|user| {msg[1]['content']} <end>"
        f"<start>|assistant| {msg[2]['content']} <end>"
    )
    return full_text

def cust_collate(batch, tokenizer = tokenizer):
  text_list = list(batch)

  inps = tokenizer(
      text_list,
      truncation = True,
      padding = True,
      return_tensors = "pt"
  )

  labels = inps["input_ids"].clone()

  seeker = torch.tensor(tokenizer.encode("|assistant|"))

  pad_token_id = tokenizer.pad_token_id

  for i in range(labels.shape[0]):

    # This is your boolean flag: start by masking everything
    inside_assistant_output = False

    j = 0
    while j < len(labels[i]):
        # Check if we just ran into the assistant marker
        if(labels[i, j] == seeker):

          inside_assistant_output = True
          labels[i, j] = -100
          j += 1
          continue

        # Apply your rules based on the boolean state
        if inside_assistant_output == False:
            # Until we meet the assistant, mask everything
            labels[i, j] = -100
        else:
            # After we meet the assistant, don't mask... UNLESS it's a pad token
            if labels[i, j] == pad_token_id:
                labels[i, j] = -100

        j += 1

  inps["labels"] = labels
  return inps

In [7]:
from torch.utils.data import DataLoader
from functools import partial

train = CustDataset(train_dataset)
test = CustDataset(test_dataset)
collate_with_tok = partial(cust_collate, tokenizer=tokenizer)

train_loader = DataLoader(train, batch_size = 4, shuffle = True, collate_fn = collate_with_tok, num_workers = 2)
test_loader = DataLoader(test, batch_size = 4, shuffle = True, collate_fn = collate_with_tok, num_workers = 2)

In [8]:
conf = LoraConfig(
    r = 8,
    lora_alpha = 20,
    target_modules = ["c_attn", "c_proj"],
    lora_dropout = 0.05,
    bias = "none",
    task_type = TaskType.CAUSAL_LM,
    modules_to_save=["wte", "lm_head"],
    ensure_weight_tying = True
)
mod = get_peft_model(model, conf)

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [9]:
base_vocab_size = 50257

wte_weight = None
lm_head_weight = None

for name, param in mod.named_parameters():
    if "wte.modules_to_save.default" in name:
        wte_weight = param
    elif "lm_head.modules_to_save.default" in name:
        lm_head_weight = param

for name, param in mod.named_parameters():
    if "wte.modules_to_save.default" in name:
        wte_weight = param
    elif "lm_head.modules_to_save.default" in name:
        lm_head_weight = param

def zero_old_token_gradients():
    # If gradients exist, wipe them out for the first 50,257 tokens
    if wte_weight is not None and wte_weight.grad is not None:
        wte_weight.grad[:base_vocab_size, :] = 0.0

    if lm_head_weight is not None and lm_head_weight.grad is not None:
        lm_head_weight.grad[:base_vocab_size, :] = 0.0

In [69]:
from tqdm import tqdm
def trainer(model, epochs, optim, train_loader):
  model.to('cuda')
  model.train()
  for epoch in range(epochs):
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")

    for idx, batch in enumerate(pbar):

      b_inp_ids = batch['input_ids'].to('cuda')
      b_attn_mask = batch['attention_mask'].to('cuda')
      b_labels = batch['labels'].to('cuda')

      optim.zero_grad()

      outputs = model(
          input_ids = b_inp_ids,
          attention_mask = b_attn_mask,
          labels = b_labels
      )

      loss = outputs.loss

      loss.backward()
      #zero_old_token_gradients()

      optim.step()

      total_loss += loss.item()
    print(total_loss)

In [83]:
from torch.optim import AdamW
#optim = AdamW(mod.parameters(), lr = 2e-4, weight_decay = 0.0)
optim = AdamW(filter(lambda p: p.requires_grad, mod.parameters()), lr = 6e-4, weight_decay = 0.0)

trainer(model = mod, epochs = 2, optim = optim, train_loader = train_loader)

Epoch 1/2: 100%|██████████| 750/750 [02:52<00:00,  4.35batch/s]


846.765248298645


Epoch 2/2: 100%|██████████| 750/750 [02:53<00:00,  4.31batch/s]

691.5649741888046


In [84]:
def infer(model, processor, test_loader):
  model.eval()
  tokenizer.padding_side = "left"
  for idx, batch in enumerate(test_loader):
    b_inp_ids = batch['input_ids'].to('cuda')
    b_attn_mask = batch['attention_mask'].to('cuda')

    """output_ids = model.generate(
        input_ids = b_inp_ids,
        attention_mask = b_attn_mask,
        max_new_tokens = 200,
        pad_token_id = processor.pad_token_id,
        eos_token_id = processor.eos_token_id,

        # --- CRITICAL FIXES FOR STRUCTURAL TASKS ---
        do_sample = False,          # Switch to greedy search for stability
        temperature = 1,         # Not used when do_sample=False
        top_k = None,               # Not used when do_sample=False
        repetition_penalty = 1.05,   # Set to 1.0 (disabled) to avoid penalizing necessary formatting tokens
    """
    #print(b_inp_ids)
    #print(b_attn_mask)
    output_ids = model.generate(
          input_ids = b_inp_ids,
          attention_mask = b_attn_mask,
          max_new_tokens = 256,              # Bumped up slightly so the email can finish
          pad_token_id = processor.pad_token_id,
          eos_token_id = processor.eos_token_id,

          # --- Decoding settings ---
          do_sample = True,
          temperature = 0.8,                 # Keeps it focused on the prompt facts
          top_p = 0.9,                        # Nucleus sampling avoids chaotic long-tail tokens
          repetition_penalty = 1.05           # Gently discourages repeating itself
      )

    print(output_ids)

    tokenizer.padding_side = "right"

    return(processor.batch_decode(output_ids, skip_special_tokens = False))

def print_out(decoded_text):
    print("=" * 60)

    # 1. Isolate the System section
    if "|system|" in decoded_text:
        # Grab everything between |system| and the next <end> tag
        system_part = decoded_text.split("|system|")[1].split("<end>")[0].strip()
        print(f"💻 \033[1mSYSTEM:\033[0m\n{system_part}\n")

    # 2. Isolate the User section
    if "|user|" in decoded_text:
        user_part = decoded_text.split("|user|")[1].split("<end>")[0].strip()
        print(f"👤 \033[1mUSER:\033[0m\n{user_part}\n")

    # 3. Isolate the Assistant section
    if "|assistant|" in decoded_text:
        assistant_part = decoded_text.split("|assistant|")[1].split("<end>")[0].strip()
        print(f"🤖 \033[1mASSISTANT (Model Output):\033[0m\n{assistant_part}")

    print("=" * 60)


In [73]:
from torch.utils.data import Dataset
import torch

class CustDataset1(Dataset):
  def __init__(self, dataset):
    self.dataset = dataset

  def __len__(self):
    return len(self.dataset)

  def __getitem__(self, idx):
    msg = self.dataset[idx]['messages']

    full_text = (
        f"<start>|system| {msg[0]['content']} <end>"
        f"<start>|user| {msg[1]['content']} <end>"
        f"<start>|assistant| "
    )
    return full_text

In [85]:
test_dataset1 = test_dataset.select(range(4))
test1 = CustDataset1(test_dataset1)
test2 = CustDataset(test_dataset1)
test_loader1 = DataLoader(test1, batch_size = 4, shuffle = True, collate_fn = collate_with_tok, num_workers = 2)
test_loader2 = DataLoader(test2, batch_size = 4, shuffle = True, collate_fn = collate_with_tok, num_workers = 2)

In [86]:
lom = infer(mod, tokenizer, test_loader1)

tensor([[50257, 50257, 50257,  ..., 50257, 50257, 50257],
        [50258, 50260,   921,  ..., 15309,   220, 50259],
        [50257, 50257, 50257,  ..., 50257, 50257, 50257],
        [50257, 50257, 50257,  ..., 50257, 50257, 50257]], device='cuda:0')


In [87]:
print("GENERATED BY AI")
print_out(lom[0])
print("ORIGINAL")
print_out(tokenizer.decode(next(iter(test_loader2))['input_ids'][3]))

GENERATED BY AI
💻 SYSTEM:
You're an AI assistant for text re-writing. Rewrite the input text to make it more friendly and approachable while maintaining its main points.

👤 USER:
Alex,

I’ve noticed that the resources you were supposed to develop for the LGBTQ+ module are still not ready. This is really holding up the project, and we’re running out of time. I feel like I’m the only one pulling my weight here. Could you please step up and get these done by the end of the week?

Thanks,
Sarah

🤖 ASSISTANT (Model Output):
Hi Alex,

I hope you're doing well! I wanted to touch base with you about the resources you were working on for the LGBTQ+ module. It seems like they haven't been fully completed yet, and I'm really hoping we can find a way to support those who need them most. I know you're juggling a lot, and I'm sure with a bit of extra focus, your project could be on a much higher quality. Could you please let me know when we might be able to finalize them by the end of this week?

Th

In [44]:
tokenizer.encode("<start>|assistant|")

[50258, 50262]